# Extension 1: Time-based validation

The main results use a random 80/20 split. Here the models are trained on the **earliest 80%** of contacts and tested on the **latest 20%**, to check whether they still work when economic conditions change.

In [1]:
import sys, json, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from src.config import RESULTS_DIR, FIG_DIR, RANDOM_STATE, TEST_SIZE
from src.data_loading import load_raw, split_xy
from src.preprocessing import clean_and_engineer
from src.models import base_models
from src.train import build_ensembles, oof_probas
from src import evaluation as ev

# Same data and same 80/20 random split as the main notebook
clean = clean_and_engineer(load_raw())
X, y = split_xy(clean)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

# Rebuild the tuned models from the saved best hyperparameters (no re-tuning needed)
summary = json.load(open(RESULTS_DIR / "summary.json"))
tuned = base_models()
for name, info in summary["tuning"].items():
    tuned[name].set_params(**info["best_params"])
ens = build_ensembles(tuned, summary["top2"])
print("Ready. Training rows:", len(X_tr), "| Test rows:", len(X_te))

Ready. Training rows: 32940 | Test rows: 8236


In [2]:
from src.time_validation import chronological_split

Xc_tr, Xc_te, yc_tr, yc_te = chronological_split(clean)
print(f"Time-based train: {len(Xc_tr):,} rows, subscription rate {yc_tr.mean():.1%}")
print(f"Time-based test : {len(Xc_te):,} rows, subscription rate {yc_te.mean():.1%}")

Time-based train: 32,941 rows, subscription rate 6.4%
Time-based test : 8,235 rows, subscription rate 30.8%


In [3]:
labels = {"rf": "Random Forest", "hgb": "Gradient Boosting", "voting": "Soft Voting"}
models = {"rf": tuned["rf"], "hgb": tuned["hgb"], "voting": ens["voting"]}
random_results = pd.read_csv(RESULTS_DIR / "test_results_tuned_threshold.csv", index_col=0)

rows = []
for name, est in models.items():
    p = clone(est).fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:, 1]
    m = ev.metrics(yc_te, p)
    rows.append({"model": labels[name], "split": "time-based", "roc_auc": m["roc_auc"],
                 "pr_auc": m["pr_auc"], "capture_top20": m["capture_top20"]})
    r = random_results.loc[name]
    rows.append({"model": labels[name], "split": "random (main results)", "roc_auc": r["roc_auc"],
                 "pr_auc": r["pr_auc"], "capture_top20": r["capture_top20"]})
time_split = pd.DataFrame(rows)
time_split.to_csv(RESULTS_DIR / "time_split_results.csv", index=False)
time_split.round(3)

,model,split,roc_auc,pr_auc,capture_top20
0,Random Forest,time-based,0.721,0.508,0.351
1,Random Forest,random (main results),0.811,0.482,0.657
2,Gradient Boosting,time-based,0.647,0.432,0.328
3,Gradient Boosting,random (main results),0.812,0.486,0.663
4,Soft Voting,time-based,0.683,0.466,0.340
5,Soft Voting,random (main results),0.813,0.487,0.661


In [4]:
pivot = time_split.pivot(index="model", columns="split", values="roc_auc")
ax = pivot.plot.bar(figsize=(8, 4.5), rot=0)
ax.set(ylabel="ROC-AUC", ylim=(0.5, 0.85), title="Random split vs time-based split (test ROC-AUC)")
ax.legend(title="")
plt.savefig(FIG_DIR / "14_time_split.png", dpi=130, bbox_inches="tight")
plt.show()